Literature Download using Elsapy

Which Literature we could use

apikey = "Get from Elsavier API key"
Search Elsapy for Keywords

In [ ]:
import time
import urllib.parse
import pandas as pd
from elsapy.elsclient import ElsClient
from elsapy.elsdoc import FullDoc
import os
apikey = os.environ["ELSEVIER_APIKEY"]
client = ElsClient(apikey)

max_papers = 500
batch_size = 25

# broad candidate search
query = """
TITLE-ABS-KEY(
    ("directed energy deposition" OR DED OR "laser directed energy deposition" OR
     "laser metal deposition" OR LMD OR LENS OR "laser engineered net shaping") AND
    (NiTi OR Nitinol OR "nickel titanium") AND
    ("shape memory alloy" OR SMA OR superelastic*)
)
"""

# category-based keywords for full-text relevance
keyword_groups = {
    "process": [
        "directed energy deposition", "ded", "laser directed energy deposition",
        "laser metal deposition", "lmd", "lens", "laser engineered net shaping",
        "powder feed rate", "mass flow rate", "scan speed", "laser power",
        "energy density", "layer thickness", "build strategy", "multi-axis", "in-situ monitoring",
        "wire-arc additive manufacturing", "waam", "electron beam additive manufacturing", "ebam"
    ],
    "material": [
        "niti", "nitinol", "nickel titanium", "shape memory alloy", "sma",
        "superelastic", "phase transformation", "transformation temperature",
        "martensite", "austenite", "ti2ni", "ni4ti3"
    ],
    "characterization": [
        "dsc", "differential scanning calorimetry", "xrd", "sem", "eds",
        "microhardness", "compression test", "tensile test"
    ],
    "outputs": [
        "microstructure", "porosity", "crack", "hardness",
        "transformation temperature", "shape memory effect", "superelasticity"
    ]
}####These were modified to be more specific to the topic and to include some common AM process parameters and characterization methods.

print("Step 1: Starting paged search...")
t0 = time.time()

all_results = []
start = 0
total_available = None

encoded_query = urllib.parse.quote(query)
base_url = "https://api.elsevier.com/content/search/scopus"

while len(all_results) < max_papers:
    url = f"{base_url}?query={encoded_query}&count={batch_size}&start={start}"
    print(f"Requesting results {start} to {start + batch_size - 1}...")

    data = client.exec_request(url)
    search_results = data.get("search-results", {})
    entries = search_results.get("entry", [])

    if total_available is None:
        total_available = int(search_results.get("opensearch:totalResults", 0))
        print("Total available in Scopus:", total_available)

    if not entries:
        print("No more results returned.")
        break

    all_results.extend(entries)
    print(f"  Got {len(entries)} results, total collected: {len(all_results)}")

    if len(entries) < batch_size:
        print("Last page reached.")
        break

    start += batch_size
    if start >= total_available:
        print("Reached total available results.")
        break

all_results = all_results[:max_papers]

print(f"Step 1 done in {time.time() - t0:.2f} sec")
print("Requested:", max_papers)
print("Collected:", len(all_results))

print("Step 2: Building search dataframe...")
search_records = []
for r in all_results:
    search_records.append({
        "title": r.get("dc:title", ""),
        "pii": r.get("pii", ""),
        "doi": r.get("prism:doi", ""),
        "eid": r.get("eid", ""),
        "publication": r.get("prism:publicationName", ""),
        "coverDate": r.get("prism:coverDate", ""),
    })

search_df = pd.DataFrame(search_records)
print("Search dataframe size:", len(search_df))
display(search_df.head())

print("Step 3: Preparing identifiers...")

records_to_check = []
seen = set()

for _, row in search_df.iterrows():
    pii = str(row["pii"]).strip() if pd.notna(row["pii"]) else ""
    doi = str(row["doi"]).strip() if pd.notna(row["doi"]) else ""

    if pii:
        key = ("pii", pii)
        if key not in seen:
            seen.add(key)
            records_to_check.append({
                "id_type": "pii",
                "id_value": pii,
                "title": row["title"],
                "doi": doi,
            })
    elif doi:
        key = ("doi", doi)
        if key not in seen:
            seen.add(key)
            records_to_check.append({
                "id_type": "doi",
                "id_value": doi,
                "title": row["title"],
                "doi": doi,
            })

print("Total search results:", len(search_df))
print("Records with PII or DOI:", len(records_to_check))

def extract_text(raw_text):
    if isinstance(raw_text, str):
        return raw_text
    if isinstance(raw_text, dict):
        texts = []

        def recurse(x):
            if isinstance(x, str):
                texts.append(x)
            elif isinstance(x, dict):
                for v in x.values():
                    recurse(v)
            elif isinstance(x, list):
                for item in x:
                    recurse(item)

        recurse(raw_text)
        return " ".join(texts)
    return ""

def score_relevance(full_text, keyword_groups):
    matched_groups = {}

    for group_name, kws in keyword_groups.items():
        hits = [kw for kw in kws if kw.lower() in full_text]
        matched_groups[group_name] = hits

    group_hit_count = sum(len(v) > 0 for v in matched_groups.values())
    total_hits = sum(len(v) for v in matched_groups.values())

    return matched_groups, group_hit_count, total_hits

def flatten_matched_groups(matched_groups):
    parts = []
    for group_name, hits in matched_groups.items():
        if hits:
            parts.append(f"{group_name}: {', '.join(hits)}")
    return " | ".join(parts)

print("Step 4: Checking full text and content relevance...")
results = []

for i, rec in enumerate(records_to_check, 1):
    item_start = time.time()
    id_type = rec["id_type"]
    id_value = rec["id_value"]

    print(f"[{i}/{len(records_to_check)}] Checking {id_type.upper()}: {id_value}")

    try:
        if id_type == "pii":
            doc = FullDoc(sd_pii=id_value)
        else:
            doc = FullDoc(doi=id_value)
    except Exception as e:
        results.append({
            "id_type": id_type,
            "id_value": id_value,
            "title": rec["title"],
            "doi": rec["doi"],
            "openaccess": "",
            "full_text_available": False,
            "matched_keywords": "",
            "group_hit_count": 0,
            "total_keyword_hits": 0,
            "relevant_in_fulltext": False,
            "reason": f"init_failed: {e}",
        })
        print(f"  -> init failed in {time.time() - item_start:.2f} sec")
        continue

    if not doc.read(client):
        results.append({
            "id_type": id_type,
            "id_value": id_value,
            "title": rec["title"],
            "doi": rec["doi"],
            "openaccess": "",
            "full_text_available": False,
            "matched_keywords": "",
            "group_hit_count": 0,
            "total_keyword_hits": 0,
            "relevant_in_fulltext": False,
            "reason": "read_failed",
        })
        print(f"  -> read failed in {time.time() - item_start:.2f} sec")
        continue

    core = doc.data.get("coredata", {})
    raw = doc.data.get("originalText", "")
    objects = doc.data.get("objects", [])

    has_fulltext_string = isinstance(raw, str) and (
        "FULL-TEXT" in raw or "Introduction" in raw or "Materials and methods" in raw
    )
    has_pdf_object = isinstance(objects, list) and any(
        "pdf" in str(obj).lower() for obj in objects
    )

    full_text_available = has_fulltext_string or has_pdf_object

    if full_text_available:
        full_text = extract_text(raw).lower()
        matched_groups, group_hit_count, total_hits = score_relevance(full_text, keyword_groups)
        matched_keywords_text = flatten_matched_groups(matched_groups)
    else:
        matched_groups = {}
        group_hit_count = 0
        total_hits = 0
        matched_keywords_text = ""

    relevant_in_fulltext = (
        full_text_available and
        group_hit_count >= 2 and
        total_hits >= 3
    )

    results.append({
        "id_type": id_type,
        "id_value": id_value,
        "title": core.get("dc:title", rec["title"]),
        "doi": core.get("prism:doi", rec["doi"]),
        "openaccess": core.get("openaccess", ""),
        "full_text_available": full_text_available,
        "matched_keywords": matched_keywords_text,
        "group_hit_count": group_hit_count,
        "total_keyword_hits": total_hits,
        "relevant_in_fulltext": relevant_in_fulltext,
        "reason": "",
    })

    print(
        f"  -> done in {time.time() - item_start:.2f} sec | "
        f"full_text={full_text_available} | "
        f"group_hits={group_hit_count} | "
        f"total_hits={total_hits}"
    )

    if i % 20 == 0:
        pd.DataFrame(results).to_csv("fulltext_check_progress.csv", index=False)
        print(f"  Progress saved at record {i}")

results_df = pd.DataFrame(results)

selected_df = results_df[
    (results_df["full_text_available"] == True) &
    (results_df["relevant_in_fulltext"] == True)
].copy()

print("\nSummary")
print("Requested papers:", max_papers)
print("Collected from search:", len(search_df))
print("Checked with PII/DOI:", len(results_df))
print("Full-text papers:", results_df["full_text_available"].sum())
print("Relevant in full text:", len(selected_df))

# search_df.to_csv("search_results_1000.csv", index=False)
# results_df.to_csv("fulltext_and_keyword_check_1000.csv", index=False)
selected_df.to_csv("selected_relevant_fulltext_papers_1000.csv", index=False)
print(f"documents: {len(selected_df)}")

print("Saved files:")
# print("- search_results_1000.csv")
# print("- fulltext_and_keyword_check_1000.csv")
print("- selected_relevant_fulltext_papers_1000.csv")
# print("- fulltext_check_progress.csv")

Step 1: Starting paged search...
Requesting results 0 to 24...
Total available in Scopus: 119
  Got 25 results, total collected: 25
Requesting results 25 to 49...
  Got 25 results, total collected: 50
Requesting results 50 to 74...
  Got 25 results, total collected: 75
Requesting results 75 to 99...
  Got 25 results, total collected: 100
Requesting results 100 to 124...
  Got 19 results, total collected: 119
Last page reached.
Step 1 done in 8.83 sec
Requested: 500
Collected: 119
Step 2: Building search dataframe...
Search dataframe size: 119


,title,pii,doi,eid,publication,coverDate
0,Synergistic strengthening for stable superelas...,S1359646226001193,10.1016/j.scriptamat.2026.117283,2-s2.0-105032871724,Scripta Materialia,2026-06-01
1,In-situ fabrication of NiTiTa alloys by laser-...,S0921509326001401,10.1016/j.msea.2026.149860,2-s2.0-105028787695,Materials Science and Engineering A,2026-03-01
2,Investigating the Effects of Build Height on M...,,10.1002/adem.202501950,2-s2.0-105024590134,Advanced Engineering Materials,2026-02-04
3,"Influence of Deposition Conditions, Powder Fee...",,10.3390/cryst16020098,2-s2.0-105031093705,Crystals,2026-02-01
4,Interface Microstructure and Properties of Inv...,,10.3788/CJL251407,2-s2.0-105031428016,Zhongguo Jiguang Chinese Journal of Lasers,2026-02-01


Step 3: Preparing identifiers...
Total search results: 119
Records with PII or DOI: 112
Step 4: Checking full text and content relevance...
[1/112] Checking PII: S1359646226001193
  -> done in 1.52 sec | full_text=True | group_hits=4 | total_hits=19
[2/112] Checking PII: S0921509326001401
  -> done in 1.61 sec | full_text=True | group_hits=4 | total_hits=31
[3/112] Checking DOI: 10.1002/adem.202501950
  -> read failed in 1.20 sec
[4/112] Checking DOI: 10.3390/cryst16020098
  -> read failed in 1.16 sec
[5/112] Checking DOI: 10.3788/CJL251407
  -> read failed in 1.19 sec
[6/112] Checking DOI: 10.3390/bioengineering13020216
  -> read failed in 1.26 sec
[7/112] Checking PII: S0264127525017769
  -> done in 1.53 sec | full_text=True | group_hits=4 | total_hits=31
[8/112] Checking DOI: 10.1002/adfm.202530524
  -> read failed in 1.14 sec
[9/112] Checking DOI: 10.1007/s11665-025-11403-2
  -> read failed in 1.16 sec
[10/112] Checking DOI: 10.1007/s40830-025-00586-1
  -> read failed in 1.18 sec
[

In [1]:
import torch
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
torch.version.cuda: 12.8
cuda available: True
device count: 1
NVIDIA GeForce RTX 5080


In [3]:
apikey = "7ce3b8cfe49f1a180f668d78658b6003"

import re
import pandas as pd
from elsapy.elsclient import ElsClient
from elsapy.elsdoc import FullDoc
import torch
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain_huggingface import HuggingFaceEmbeddings
from llama_index.llms.ollama import Ollama

client = ElsClient(apikey)

input_file = "selected_relevant_fulltext_papers_1000.xlsx"
df = pd.read_excel(input_file).head(30)

def extract_text(raw_text):
    if isinstance(raw_text, str):
        return raw_text
    if isinstance(raw_text, dict):
        texts = []

        def recurse(x):
            if isinstance(x, str):
                texts.append(x)
            elif isinstance(x, dict):
                for v in x.values():
                    recurse(v)
            elif isinstance(x, list):
                for item in x:
                    recurse(item)

        recurse(raw_text)
        return " ".join(texts)
    return ""

def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()

def split_windows(text, window_size=1800, step=1200):
    text = normalize_text(text)
    if not text:
        return []

    if len(text) <= window_size:
        return [(0, text)]

    windows = []
    starts = list(range(0, len(text) - window_size + 1, step))
    if starts[-1] != len(text) - window_size:
        starts.append(len(text) - window_size)

    for start in starts:
        windows.append((start, text[start:start + window_size]))
    return windows

def bad_section(chunk):
    t = chunk.lower()
    bad_markers = [
        "references",
        "acknowledgement",
        "acknowledgment",
        "declaration of competing interest",
        "declarations",
        "credit authorship contribution statement",
        "funding",
        "data availability",
        "conflict of interest",
        "open access funding",
    ]
    return any(marker in t for marker in bad_markers)

section_keywords = {
    "methods_section": [
        "materials and methods",
        "experimental procedure",
        "experimental setup",
        "methodology",
        "sample preparation",
        "welding procedure",
        "process parameters",
        "experimental details",
        "materials",
        "methods"
    ],
    "equipment": [
        "welding machine",
        "power source",
        "torch",
        "thermocouple",
        "sem",
        "scanning electron microscope",
        "eds",
        "xrd",
        "microhardness",
        "hardness test",
        "tensile test",
        "tensile testing",
        "impact test",
        "charpy"
    ],
    "process": [
        "heat input",
        "cooling rate",
        "current",
        "voltage",
        "travel speed",
        "welding speed",
        "shielding gas",
        "interpass temperature",
        "preheat",
        "post weld heat treatment",
        "filler metal",
        "weld metal",
        "base metal",
        "gmaw",
        "gtaw",
        "saw",
        "fcaw"
    ],
    "outputs": [
        "yield strength",
        "ultimate tensile strength",
        "toughness",
        "microstructure",
        "hardness",
        "residual stress",
        "grain size",
        "phase transformation",
        "bainite",
        "martensite",
        "acicular ferrite",
        "carbides",
        "cct",
        "ttt"
    ]
}

def count_group_hits(text, keyword_dict):
    out = {}
    for group, kws in keyword_dict.items():
        hits = [kw for kw in kws if kw.lower() in text]
        out[group] = hits
    return out

def flatten_hits(hit_dict):
    parts = []
    for group, hits in hit_dict.items():
        if hits:
            parts.append(f"{group}: {', '.join(hits)}")
    return " | ".join(parts)

def chunk_quality_stats(chunk):
    t = chunk.lower()

    words = len(t.split())
    dots = t.count(".")
    commas = t.count(",")

    bad = (
        t.count("http") +
        t.count(".svg") +
        t.count(".jpg") +
        t.count(".jpeg") +
        t.count(".png") +
        t.count("altimg") * 2 +
        t.count("image/svg+xml") +
        t.count("image/jpeg") +
        t.count("image-high-res") +
        t.count("fig.") +
        t.count("table ") +
        t.count("doi") * 2 +
        t.count("references") * 3
    )

    citation_like = (
        t.count(" et al.") +
        t.count(" doi:") +
        t.count(" vol.") +
        t.count(" pp.")
    )

    digits = sum(c.isdigit() for c in t)
    letters = sum(c.isalpha() for c in t)
    digit_ratio = digits / max(len(t), 1)
    letter_ratio = letters / max(len(t), 1)

    return {
        "words": words,
        "dots": dots,
        "commas": commas,
        "bad": bad,
        "citation_like": citation_like,
        "digit_ratio": digit_ratio,
        "letter_ratio": letter_ratio,
    }

def score_chunk(chunk):
    t = chunk.lower()
    stats = chunk_quality_stats(chunk)

    if bad_section(chunk):
        return {
            "keep": False,
            "score": -999,
            "reason": "bad_section",
            "matched_groups": {},
            "group_hit_count": 0,
            "total_hits": 0,
        }

    matched_groups = count_group_hits(t, section_keywords)
    group_hit_count = sum(len(v) > 0 for v in matched_groups.values())
    total_hits = sum(len(v) for v in matched_groups.values())

    score = 0

    if stats["words"] >= 80:
        score += 1
    if stats["words"] >= 140:
        score += 1
    if stats["dots"] >= 3:
        score += 1
    if stats["commas"] >= 2:
        score += 1
    if stats["bad"] <= 6:
        score += 1
    if stats["citation_like"] <= 5:
        score += 1
    if stats["digit_ratio"] <= 0.25:
        score += 1
    if stats["letter_ratio"] >= 0.50:
        score += 1

    score += group_hit_count * 2
    score += min(total_hits, 6)

    if len(matched_groups["methods_section"]) > 0:
        score += 3
    if len(matched_groups["equipment"]) > 0:
        score += 2
    if len(matched_groups["process"]) > 0:
        score += 2
    if len(matched_groups["outputs"]) > 0:
        score += 1

    keep = (
        group_hit_count >= 2 and
        total_hits >= 3 and
        score >= 10
    )

    if len(matched_groups["methods_section"]) > 0 and len(matched_groups["process"]) > 0 and total_hits >= 3:
        keep = True

    return {
        "keep": keep,
        "score": score,
        "reason": "keep" if keep else "low_score_or_low_relevance",
        "matched_groups": matched_groups,
        "group_hit_count": group_hit_count,
        "total_hits": total_hits,
    }

documents = []
debug_rows = []

for i, row in df.iterrows():
    title = str(row["title"]).strip()
    id_type = str(row["id_type"]).strip().lower()
    id_value = str(row["id_value"]).strip()
    doi = str(row["doi"]).strip() if "doi" in row and pd.notna(row["doi"]) else ""

    print(f"[{i+1}/{len(df)}] Checking: {title}")

    try:
        doc = FullDoc(sd_pii=id_value) if id_type == "pii" else FullDoc(doi=id_value)
    except Exception as e:
        debug_rows.append({
            **row.to_dict(),
            "status": "drop",
            "reason": f"init_failed: {e}",
            "good_chunk_count": 0,
            "best_chunk_score": None,
            "best_chunk_hits": "",
            "preview": "",
        })
        print("  init failed")
        continue

    if not doc.read(client):
        debug_rows.append({
            **row.to_dict(),
            "status": "drop",
            "reason": "read_failed",
            "good_chunk_count": 0,
            "best_chunk_score": None,
            "best_chunk_hits": "",
            "preview": "",
        })
        print("  read failed")
        continue

    raw_text = doc.data.get("originalText", "")
    text = normalize_text(extract_text(raw_text))

    windows = split_windows(text, window_size=1800, step=1200)
    selected_chunks = []
    scored_chunks = []

    for start, chunk in windows:
        result = score_chunk(chunk)
        scored_chunks.append((start, chunk, result))
        if result["keep"]:
            selected_chunks.append((start, chunk, result))

    if selected_chunks:
        for j, (start, chunk, result) in enumerate(selected_chunks):
            documents.append(
                Document(
                    text=chunk,
                    metadata={
                        "title": title,
                        "id_type": id_type,
                        "id_value": id_value,
                        "doi": doi,
                        "chunk_start": start,
                        "chunk_id": j,
                        "group_hit_count": result["group_hit_count"],
                        "total_hits": result["total_hits"],
                        "matched_groups": flatten_hits(result["matched_groups"]),
                    }
                )
            )

        best_chunk = max(selected_chunks, key=lambda x: x[2]["score"])
        best_hits_text = flatten_hits(best_chunk[2]["matched_groups"])

        debug_rows.append({
            **row.to_dict(),
            "status": "keep",
            "reason": "has_relevant_chunks",
            "good_chunk_count": len(selected_chunks),
            "best_chunk_score": best_chunk[2]["score"],
            "best_chunk_hits": best_hits_text,
            "preview": best_chunk[1][:1000],
        })
        print(f"  keep | relevant chunks: {len(selected_chunks)}")
    else:
        if scored_chunks:
            best_chunk = max(scored_chunks, key=lambda x: x[2]["score"])
            best_hits_text = flatten_hits(best_chunk[2]["matched_groups"])
            preview_text = best_chunk[1][:1000]
            best_score = best_chunk[2]["score"]
            reason = best_chunk[2]["reason"]
        else:
            best_hits_text = ""
            preview_text = text[:1000]
            best_score = None
            reason = "no_windows"

        debug_rows.append({
            **row.to_dict(),
            "status": "drop",
            "reason": reason,
            "good_chunk_count": 0,
            "best_chunk_score": best_score,
            "best_chunk_hits": best_hits_text,
            "preview": preview_text,
        })
        print("  drop | no relevant chunks")

debug_df = pd.DataFrame(debug_rows)
debug_df.to_excel("chunk_filter_debug.xlsx", index=False)

print("\nTotal selected chunks:", len(documents))
print("Saved debug file: chunk_filter_debug.xlsx")

query_text = """
Find literature chunks that describe one welding-related experiment,
especially:
- experimental procedure or setup
- equipment used
- welding process parameters
- material system
- measurements or outputs
for welded steel or low alloy steel.
"""

if documents:

    Settings.embed_model = LangchainEmbedding(
        HuggingFaceEmbeddings(
            model_name="BAAI/bge-small-en-v1.5",
            model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
    )
    Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
    
    index = VectorStoreIndex.from_documents(documents)

    retriever = index.as_retriever(similarity_top_k=4)
    retrieved_nodes = retriever.retrieve(query_text)

    context_blocks = []

    for i, node in enumerate(retrieved_nodes, 1):
        title = node.metadata.get("title")
        doi = node.metadata.get("doi")
        chunk_start = node.metadata.get("chunk_start")
        matched_groups = node.metadata.get("matched_groups")
        chunk_text = node.text

        print(f"\n--- Retrieved chunk {i} ---")
        print("title:", title)
        print("doi:", doi)
        print("chunk_start:", chunk_start)
        print("matched_groups:", matched_groups)
        print(chunk_text)

        context_blocks.append(
            f"""[Retrieved Chunk {i}]
Title: {title}
DOI: {doi}
Chunk Start: {chunk_start}
Matched Groups: {matched_groups}

Text:
{chunk_text}"""
        )

    context_text = "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(context_blocks)

    prompt = f"""
Use ONLY the retrieved chunks below.

{context_text}

Task:
Identify one welding-related experiment described in the retrieved chunks
and summarize it in this format:

- Proposed Experiment:
- Needed Equipment:
- Key Process Parameters:
- Measurements / Outputs:
- Importance of this Experiment:
- Source Document Title:

Rules:
- Do not invent authors, years, citations, equipment, parameters, or source titles.
- Use only information explicitly supported by the retrieved chunks above.
- If the retrieved chunks are insufficient to specify equipment, procedure, parameters, or measurements, say so explicitly.
- Choose the experiment from ONLY ONE source document.
- Use the exact source document title from the retrieved chunk metadata.
- Do not combine information from multiple documents.
"""

    response = Settings.llm.complete(prompt)

    print("\nResponse:")
    print(response.text if hasattr(response, "text") else str(response))
else:
    print("\nNo relevant chunks found.")

[1/30] Checking: Synergistic strengthening for stable superelasticity in additively manufactured coarse-grained NiTi Alloys
  keep | relevant chunks: 3
[2/30] Checking: In-situ fabrication of NiTiTa alloys by laser-directed energy deposition: Strength-toughness synergy mechanisms
  keep | relevant chunks: 28
[3/30] Checking: Laser directed energy deposition of NiTi shape memory alloys with herringbone grain architectures for enhanced ductility and superelasticity
  keep | relevant chunks: 23
[4/30] Checking: Microstructure and elastocaloric properties of in-situ NiTiFe alloys prepared via laser engineered net shaping
  keep | relevant chunks: 15
[5/30] Checking: Precipitate-induced stress field steering martensite nucleation in NiTi alloys fabricated by laser direct energy deposition
  keep | relevant chunks: 16
[6/30] Checking: Hard pseudoelastic TiNiCu shape memory alloy development via a novel thermocapillary-driven additive manufacturing route
  keep | relevant chunks: 21
[7/30] Ch

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Retrieved chunk 1 ---
title: Laser metal deposition of NiTi shape memory alloy: Processability, microstructure, and thermomechanical evaluation
doi: 10.1016/j.jmrt.2025.06.029
chunk_start: 24000
matched_groups: equipment: sem, eds, xrd, hardness test, tensile test, tensile testing | process: voltage | outputs: hardness, martensite
shows the analysis at the outer edge of the sample; and (d) presents the averaged composition result from the line scan indicated in (c). Fig. 16 Fig. 17 Visualization of the ratio of Ni and Ti content across the layers and weld lines between the layers. Fig. 17 Fig. 18 XRD results performed at a) 30 °C and b) 150 °C temperatures. The measurements are shown together with the calculated profiles from the Rietveld refinement. The residual for the refinement is shown by the grey line at the bottom of the graph. It should be noted that the NiTi2 phase is also present at 30 °C but obscured by the abundance of peaks caused by the monoclinic B19′ martensite. Fi

In [7]:
import json
import re
from pathlib import Path
from collections import defaultdict

from llama_index.core import Document, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.core import Document, VectorStoreIndex, Settings
Settings.llm = Ollama(model="qwen3:8b", request_timeout=180.0, context_window=4096)

OUT_DIR = Path("paper_card_store")
OUT_DIR.mkdir(exist_ok=True)

def extract_json_object(text: str):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m.group(0))

# Group selected chunks by paper
paper_chunks = defaultdict(list)
for d in documents:
    key = (d.metadata.get("title", ""), d.metadata.get("doi", ""))
    paper_chunks[key].append(d)

paper_cards = []
paper_card_docs = []

CARD_TEMPLATE = {
    "source_title": "",
    "doi": "",
    "process_family": "unknown",
    "material_family": "unknown",
    "material_system": [],
    "feedstock_form": [],
    "substrate": [],
    "equipment": [],
    "consumables": [],
    "controllable_parameters": [],
    "measurements_outputs": [],
    "heat_treatment": [],
    "microstructure_terms": [],
    "bom_keywords": [],
    "experiment_summary": "",
    "unknowns": [],
    "feasibility_notes": "",
}

LIST_FIELDS = {
    "material_system",
    "feedstock_form",
    "substrate",
    "equipment",
    "consumables",
    "controllable_parameters",
    "measurements_outputs",
    "heat_treatment",
    "microstructure_terms",
    "bom_keywords",
    "unknowns",
}

STRING_FIELDS = {
    "source_title",
    "doi",
    "process_family",
    "material_family",
    "experiment_summary",
    "feasibility_notes",
}

def normalize_card(card, title, doi):
    out = CARD_TEMPLATE.copy()

    if isinstance(card, dict):
        for k, v in card.items():
            if k in out:
                out[k] = v

    out["source_title"] = title
    out["doi"] = doi

    for k in LIST_FIELDS:
        v = out.get(k, [])
        if v is None:
            out[k] = []
        elif isinstance(v, str):
            out[k] = [v.strip()] if v.strip() else []
        elif isinstance(v, list):
            out[k] = [str(x).strip() for x in v if str(x).strip()]
        else:
            out[k] = [str(v).strip()]

    for k in STRING_FIELDS:
        v = out.get(k, "")
        if v is None:
            out[k] = ""
        else:
            out[k] = str(v).strip()

    return out

for paper_idx, ((title, doi), chunk_docs) in enumerate(paper_chunks.items(), 1):
    print(f"\nProcessing paper {paper_idx}/{len(paper_chunks)}: {title} | DOI: {doi} | Chunks: {len(chunk_docs)}")
    chunk_docs = sorted(
        chunk_docs,
        key=lambda d: (
            d.metadata.get("group_hit_count", 0),
            d.metadata.get("total_hits", 0),
        ),
        reverse=True,
    )

    evidence_blocks = []
    for i, d in enumerate(chunk_docs[:6], 1):
        evidence_blocks.append(
            f"""[Evidence Chunk {i}]
Title: {title}
DOI: {doi}
Chunk Start: {d.metadata.get("chunk_start")}
Matched Groups: {d.metadata.get("matched_groups")}

Text:
{d.text}
"""
        )

    evidence_text = "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(evidence_blocks)

    prompt = f"""
You are creating ONE structured experiment card from ONE paper.

Use ONLY the evidence below.
Do not invent facts.
If a field is not explicitly supported, use an empty list or "unknown".

Evidence:
{evidence_text}

Return ONLY valid JSON with this schema:

{{
  "source_title": "",
  "doi": "",
  "process_family": "",
  "material_family": "",
  "material_system": [],
  "feedstock_form": [],
  "substrate": [],
  "equipment": [],
  "consumables": [],
  "controllable_parameters": [],
  "measurements_outputs": [],
  "heat_treatment": [],
  "microstructure_terms": [],
  "bom_keywords": [],
  "experiment_summary": "",
  "unknowns": [],
  "feasibility_notes": ""
}}

Guidance:
- process_family should be something coarse like "GMAW", "GTAW", "DED", "WAAM", "LPBF", "friction stir welding"
- material_family should be coarse like "steel", "low alloy steel", "NiTi", "aluminum", "nickel alloy"
- material_system is the specific material names/compositions mentioned
- bom_keywords should be concrete nouns someone could match against an available bill of materials
- experiment_summary should be a short factual summary of what the paper actually did
"""
    print(f"  Sending {min(len(chunk_docs), 6)} evidence chunks to LLM...")
    raw = Settings.llm.complete(prompt)
    print("  LLM response received.")
    raw_text = raw.text if hasattr(raw, "text") else str(raw)

    try:
        card = extract_json_object(raw_text)
    except Exception:
        card = {}
    missing_keys = [k for k in CARD_TEMPLATE if k not in card]
    if missing_keys:
        print(f"  Missing keys from LLM output: {missing_keys}")
    card = normalize_card(card, title, doi)

    card_text = f"""
Title: {card.get('source_title', '')}
DOI: {card.get('doi', '')}
Process Family: {card.get('process_family', 'unknown')}
Material Family: {card.get('material_family', 'unknown')}
Material System: {", ".join(card.get('material_system', []))}
Feedstock Form: {", ".join(card.get('feedstock_form', []))}
Substrate: {", ".join(card.get('substrate', []))}
Equipment: {", ".join(card.get('equipment', []))}
Consumables: {", ".join(card.get('consumables', []))}
Controllable Parameters: {", ".join(card.get('controllable_parameters', []))}
Measurements / Outputs: {", ".join(card.get('measurements_outputs', []))}
Heat Treatment: {", ".join(card.get('heat_treatment', []))}
Microstructure Terms: {", ".join(card.get('microstructure_terms', []))}
BOM Keywords: {", ".join(card.get('bom_keywords', []))}

Experiment Summary:
{card.get('experiment_summary', '')}

Feasibility Notes:
{card.get('feasibility_notes', '')}
""".strip()
    
    paper_card_docs.append(
        Document(
            text=card_text,
            metadata={
                "source_title": card["source_title"],
                "doi": card["doi"],
                "process_family": card["process_family"],
                "material_family": card["material_family"],
                "material_system_str": " | ".join(card["material_system"]),
                "equipment_str": " | ".join(card["equipment"]),
                "bom_keywords_str": " | ".join(card["bom_keywords"]),
            },
        )
    )
    print(
        "  Card normalized | "
        f"process_family={card.get('process_family')} | "
        f"material_family={card.get('material_family')} | "
        f"equipment={len(card.get('equipment', []))} | "
        f"outputs={len(card.get('measurements_outputs', []))}"
    )
    paper_cards.append(card)
    

jsonl_path = OUT_DIR / "paper_cards.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for card in paper_cards:
        f.write(json.dumps(card, ensure_ascii=False) + "\n")

paper_card_index = VectorStoreIndex.from_documents(paper_card_docs, show_progress=True)
paper_card_index.storage_context.persist(persist_dir=str(OUT_DIR / "storage"))

print(f"Saved {len(paper_cards)} paper cards to {jsonl_path}")
print(f"Persisted vector store to {OUT_DIR / 'storage'}")


Processing paper 1/30: Synergistic strengthening for stable superelasticity in additively manufactured coarse-grained NiTi Alloys | DOI: 10.1016/j.scriptamat.2026.117283 | Chunks: 3
  Sending 3 evidence chunks to LLM...
  LLM response received.
  Card normalized | process_family=additive manufacturing | material_family=NiTi | equipment=4 | outputs=4

Processing paper 2/30: In-situ fabrication of NiTiTa alloys by laser-directed energy deposition: Strength-toughness synergy mechanisms | DOI: 10.1016/j.msea.2026.149860 | Chunks: 28
  Sending 6 evidence chunks to LLM...
  LLM response received.
  Card normalized | process_family=DED | material_family=NiTi | equipment=9 | outputs=8

Processing paper 3/30: Laser directed energy deposition of NiTi shape memory alloys with herringbone grain architectures for enhanced ductility and superelasticity | DOI: 10.1016/j.matdes.2025.115355 | Chunks: 23
  Sending 6 evidence chunks to LLM...
  LLM response received.
  Card normalized | process_family=D

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/30 [00:00<?, ?it/s]

Saved 30 paper cards to paper_card_store/paper_cards.jsonl
Persisted vector store to paper_card_store/storage


In [8]:
import json
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterCondition

available_bom = {
    "material_family": "steel",
    "process_family": "GMAW",
    "materials": [
        "HSLA steel plate",
        "ER70S-6 wire",
        "argon",
        "thermocouple",
    ],
    "equipment": [
        "GMAW power source",
        "CNC table",
        "thermocouple",
        "SEM",
        "microhardness tester",
        "tensile testing machine",
    ],
    "forbidden_items": [
        "XRD",
        "EBSD",
    ],
    "goal": "Propose one feasible welding experiment that studies how heat input and interpass temperature affect microstructure and mechanical properties."
}

filters = MetadataFilters(
    filters=[
        MetadataFilter(key="process_family", value=available_bom["process_family"]),
        MetadataFilter(key="material_family", value=available_bom["material_family"]),
    ],
    condition=FilterCondition.AND,
)

retriever = paper_card_index.as_retriever(similarity_top_k=8, filters=filters)
retrieved_cards = retriever.retrieve(available_bom["goal"])

def overlap_score(node, bom):
    text = (node.text or "").lower()
    score = 0

    for item in bom["materials"]:
        if item.lower() in text:
            score += 2
    for item in bom["equipment"]:
        if item.lower() in text:
            score += 2
    for item in bom["forbidden_items"]:
        if item.lower() in text:
            score -= 4

    return score

ranked = sorted(
    retrieved_cards,
    key=lambda n: overlap_score(n, available_bom),
    reverse=True,
)

shortlist = ranked[:4]

print("Shortlisted paper cards:")
for i, node in enumerate(shortlist, 1):
    print(f"\n[{i}] {node.metadata.get('source_title')}")
    print(f"process_family={node.metadata.get('process_family')}")
    print(f"material_family={node.metadata.get('material_family')}")
    print(node.text[:1000], "...\n")

card_blocks = []
for i, node in enumerate(shortlist, 1):
    card_blocks.append(
        f"""[Paper Card {i}]
Title: {node.metadata.get('source_title')}
DOI: {node.metadata.get('doi')}

{node.text}
"""
    )

card_context = "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(card_blocks)

proposal_prompt = f"""
You are proposing ONE candidate new experiment.

Use the literature cards below as precedent, but do NOT pretend the experiment is already published.
You may synthesize across multiple papers.
You must respect the constrained BOM.

Available BOM / Lab Capability:
{json.dumps(available_bom, indent=2)}

Literature Cards:
{card_context}

Task:
Propose ONE feasible experiment that is inspired by the retrieved papers and is compatible with the BOM.

Return in this exact format:

- Candidate Experiment:
- Why This Is Feasible With Current BOM:
- Borrowed Literature Precedents:
- New Adaptation / Novel Twist:
- Needed Equipment:
- Needed Materials / Consumables:
- Key Process Parameters To Sweep:
- Measurements / Outputs:
- Main Risk / Failure Mode:
- Missing Capability / Assumption:
- Source Papers Used:

Rules:
- Do not claim the experiment is novel with certainty.
- Distinguish clearly between literature-backed elements and your proposed adaptation.
- Do not include equipment or materials not supported by the BOM unless you put them under "Missing Capability / Assumption".
- If the BOM is insufficient, say so explicitly.
- Cite source papers by exact title from the retrieved cards.
"""

proposal = Settings.llm.complete(proposal_prompt)
print(proposal.text if hasattr(proposal, "text") else str(proposal))

Shortlisted paper cards:
- Candidate Experiment:  
  Investigate the combined effects of heat input (controlled via voltage and wire feed speed) and interpass temperature (regulated by dwell time between weld passes) on the microstructure and mechanical properties of HSLA steel weldments using GMAW.  

- Why This Is Feasible With Current BOM:  
  The BOM supports all required equipment (GMAW power source, CNC table, thermocouple, SEM, microhardness tester, tensile testing machine) and materials (HSLA steel, ER70S-6 wire, argon). Interpass temperature can be indirectly controlled by adjusting the time between weld passes, and heat input is directly adjustable via process parameters.  

- Borrowed Literature Precedents:  
  - "Effect of Heat Input on Microstructure and Tensile Properties of HSLA Steel Weldments" (controls heat input via voltage/velocity).  
  - "Interpass Temperature Influence on Solidification Microstructure in GMAW" (uses thermocouples to monitor interpass temperature)